In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# 1. Load input files
# -----------------------------
data_dir = Path("data/clean")
output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

prices = pd.read_csv(data_dir / "prices.csv")
trades = pd.read_csv(data_dir / "trades.csv")
fees = pd.read_csv(data_dir / "fees.csv")
positions = pd.read_csv(data_dir / "positions.csv")

# Parse dates
prices["date"] = pd.to_datetime(prices["date"])
trades["date"] = pd.to_datetime(trades["date"])
positions["date"] = pd.to_datetime(positions["date"])

# -----------------------------
# 2. Merge trades with fees
# -----------------------------
trades = trades.merge(fees, on="trade_id", how="left")

# Signed quantity: BUY = positive, SELL = negative
trades["signed_qty"] = np.where(trades["side"] == "BUY", trades["quantity"], -trades["quantity"])

# Sort trades so processing is deterministic
trades = trades.sort_values(["date", "instrument", "trade_id"]).reset_index(drop=True)

# -----------------------------
# 3. Helper function to process one trade
# -----------------------------
def process_trade(position_qty, avg_cost, signed_qty, trade_price):
    """
    Updates position and average cost after one trade.
    Returns:
        new_position_qty
        new_avg_cost
        realized_pnl_from_trade
    """
    realized = 0.0

    # No existing position
    if position_qty == 0:
        new_position = signed_qty
        new_avg_cost = trade_price if new_position != 0 else np.nan
        return new_position, new_avg_cost, realized

    # Trade is in same direction as current position -> increase/open more
    if np.sign(position_qty) == np.sign(signed_qty):
        new_position = position_qty + signed_qty
        new_avg_cost = (
            (abs(position_qty) * avg_cost + abs(signed_qty) * trade_price)
            / abs(new_position)
        )
        return new_position, new_avg_cost, realized

    # Trade is in opposite direction -> reduce or flip position
    closing_qty = min(abs(position_qty), abs(signed_qty))

    # Long position being sold
    if position_qty > 0 and signed_qty < 0:
        realized = closing_qty * (trade_price - avg_cost)

    # Short position being bought back
    elif position_qty < 0 and signed_qty > 0:
        realized = closing_qty * (avg_cost - trade_price)

    new_position = position_qty + signed_qty

    # Fully closed
    if new_position == 0:
        new_avg_cost = np.nan

    # Reduced but still same direction as old position
    elif np.sign(new_position) == np.sign(position_qty):
        new_avg_cost = avg_cost

    # Flipped direction
    else:
        new_avg_cost = trade_price

    return new_position, new_avg_cost, realized

# -----------------------------
# 4. Build expected P&L
# -----------------------------
# Merge prices + positions so each day/instrument has one row
daily_base = (
    prices.merge(positions, on=["date", "instrument"], how="inner")
    .sort_values(["date", "instrument"])
    .reset_index(drop=True)
)

instruments = daily_base["instrument"].unique()

# Running state per instrument
state = {
    instrument: {
        "position": 0,
        "avg_cost": np.nan
    }
    for instrument in instruments
}

daily_records = []
trade_detail_records = []

for _, row in daily_base.iterrows():
    date = row["date"]
    instrument = row["instrument"]
    prior_close = row["prior_close_price"]
    close_price = row["close_price"]

    expected_start_position = row["start_position"]
    expected_end_position = row["end_position"]

    # Pull state from prior day
    start_position = state[instrument]["position"]
    avg_cost_start = state[instrument]["avg_cost"]

    # Validate against positions.csv
    if start_position != expected_start_position:
        raise ValueError(
            f"Start position mismatch for {instrument} on {date.date()}: "
            f"state={start_position}, positions.csv={expected_start_position}"
        )

    # Beginning unrealized P&L
    if start_position == 0 or pd.isna(avg_cost_start):
        beginning_unrealized = 0.0
    else:
        beginning_unrealized = start_position * (prior_close - avg_cost_start)

    # All trades for this date/instrument
    day_trades = trades[
        (trades["date"] == date) &
        (trades["instrument"] == instrument)
    ].copy()

    position_qty = start_position
    avg_cost = avg_cost_start
    realized_pnl = 0.0
    total_fees = day_trades["fee_amount"].sum() if not day_trades.empty else 0.0

    # Process trades one by one
    for _, trade in day_trades.iterrows():
        old_position = position_qty
        old_avg_cost = avg_cost

        position_qty, avg_cost, trade_realized = process_trade(
            position_qty=position_qty,
            avg_cost=avg_cost,
            signed_qty=trade["signed_qty"],
            trade_price=trade["trade_price"]
        )

        realized_pnl += trade_realized

        trade_detail_records.append({
            "date": date,
            "instrument": instrument,
            "trade_id": trade["trade_id"],
            "side": trade["side"],
            "quantity": trade["quantity"],
            "trade_price": trade["trade_price"],
            "fee_amount": trade["fee_amount"],
            "position_before": old_position,
            "avg_cost_before": old_avg_cost,
            "position_after": position_qty,
            "avg_cost_after": avg_cost,
            "realized_pnl_from_trade": trade_realized
        })

    end_position = position_qty
    avg_cost_end = avg_cost

    # Validate against positions.csv
    if end_position != expected_end_position:
        raise ValueError(
            f"End position mismatch for {instrument} on {date.date()}: "
            f"calculated={end_position}, positions.csv={expected_end_position}"
        )

    # Ending unrealized P&L
    if end_position == 0 or pd.isna(avg_cost_end):
        ending_unrealized = 0.0
    else:
        ending_unrealized = end_position * (close_price - avg_cost_end)

    unrealized_change_pnl = ending_unrealized - beginning_unrealized
    expected_total_pnl = realized_pnl + unrealized_change_pnl - total_fees

    # Optional attribution view (useful later for dashboard)
    position_pnl = start_position * (close_price - prior_close)
    trade_day_pnl = expected_total_pnl - position_pnl

    daily_records.append({
        "date": date,
        "instrument": instrument,
        "start_position": start_position,
        "end_position": end_position,
        "prior_close_price": prior_close,
        "close_price": close_price,
        "avg_cost_start": avg_cost_start,
        "avg_cost_end": avg_cost_end,
        "beginning_unrealized": beginning_unrealized,
        "ending_unrealized": ending_unrealized,
        "realized_pnl": realized_pnl,
        "unrealized_change_pnl": unrealized_change_pnl,
        "fees": total_fees,
        "position_pnl": position_pnl,
        "trade_day_pnl": trade_day_pnl,
        "expected_total_pnl": expected_total_pnl,
        "trade_count": len(day_trades)
    })

    # Update state for next day
    state[instrument]["position"] = end_position
    state[instrument]["avg_cost"] = avg_cost_end

# -----------------------------
# 5. Create output DataFrames
# -----------------------------
expected_pnl = pd.DataFrame(daily_records)
trade_detail = pd.DataFrame(trade_detail_records)

# Round numeric columns for readability
numeric_cols_expected = expected_pnl.select_dtypes(include=[np.number]).columns
expected_pnl[numeric_cols_expected] = expected_pnl[numeric_cols_expected].round(2)

if not trade_detail.empty:
    numeric_cols_trade = trade_detail.select_dtypes(include=[np.number]).columns
    trade_detail[numeric_cols_trade] = trade_detail[numeric_cols_trade].round(5)

# -----------------------------
# 6. Save outputs
# -----------------------------
expected_pnl.to_csv(output_dir / "expected_pnl.csv", index=False)
trade_detail.to_csv(output_dir / "trade_pnl_detail.csv", index=False)

print("Step 4 complete.")
print(f"expected_pnl rows: {len(expected_pnl)}")
print(f"trade_pnl_detail rows: {len(trade_detail)}")
print("\nSample expected_pnl output:")
print(expected_pnl.head(10))

Step 4 complete.
expected_pnl rows: 135
trade_pnl_detail rows: 200

Sample expected_pnl output:
        date instrument  start_position  end_position  prior_close_price  \
0 2025-01-02     AUDUSD               0       -125000               0.68   
1 2025-01-02     EURUSD               0             0               1.08   
2 2025-01-02     GBPUSD               0       -150000               1.27   
3 2025-01-03     AUDUSD         -125000       -125000               0.68   
4 2025-01-03     EURUSD               0             0               1.09   
5 2025-01-03     GBPUSD         -150000       -150000               1.27   
6 2025-01-06     AUDUSD         -125000       -200000               0.68   
7 2025-01-06     EURUSD               0       -125000               1.09   
8 2025-01-06     GBPUSD         -150000       -200000               1.26   
9 2025-01-07     AUDUSD         -200000       -250000               0.68   

   close_price  avg_cost_start  avg_cost_end  beginning_unrealized 